# 02_core_functions
Functions used by `03_main`, grouped by stage: **copy → unzip → ingest**.

## Common

In [0]:
def get_active_control_entries():
    return spark.sql(f"""
        SELECT s_no, zip_source, zip_source_path, zip_tgt_path, unzip_tgt_path,
               tgt_catalog, tgt_schema, tgt_table, is_full_load, is_incremental_load
        FROM {control_table}
        WHERE is_active = 1
        ORDER BY s_no
    """).collect()


def parse_file_name(file_name):
    """'case_data_001_20220605_20220620_20260101_235954.zip'
       -> {'table_name': 'case_data_001', 'start_date': '20220605', 'mid_date': '20220620', 'end_date': '20260101'}
       Returns None if the name does not follow the expected pattern."""
    m = FILE_NAME_PATTERN.match(file_name)
    return m.groupdict() if m else None


def write_logs(log_records, run_id):
    if not log_records:
        return
    rows = [
        (run_id, r.get("stage"),
         int(r["s_no"]) if r.get("s_no") is not None else None,
         r.get("source"), r.get("file_name"), r.get("source_path"), r.get("target_path"),
         r.get("status"), r.get("message"), r.get("start_time"), r.get("end_time"))
        for r in log_records
    ]
    spark.createDataFrame(rows, schema=LOG_SCHEMA).write.format("delta").mode("append").saveAsTable(log_table)

## Stage 1 - Copy

List the `.zip` files at the source.

- `is_full_load = 1` -> every zip is copied (this is what the first run uses).
- `is_incremental_load = 1` -> only zips whose **end date** in the file name is today's date.

In [0]:
def list_zip_files(source_path):
    source_path = source_path.rstrip("/") + "/"
    return [f for f in dbutils.fs.ls(source_path) if not f.isDir() and f.name.lower().endswith(".zip")]


def filter_zip_files_by_load_type(zip_files, is_full_load, is_incremental_load):
    if is_full_load == 1:
        return zip_files
    if is_incremental_load == 1:
        today = datetime.now().strftime("%Y%m%d")
        return [f for f in zip_files if (parse_file_name(f.name) or {}).get("end_date") == today]
    return []

Copy one zip into `<zip_tgt_path>/<mid_date>/` and return a log record.

In [0]:
def copy_zip_file(file_info, target_base_path, s_no, source):
    start_time = datetime.now()
    target_base_path = target_base_path.rstrip("/") + "/"
    record = {"s_no": s_no, "stage": "COPY", "source": source, "file_name": file_info.name,
              "source_path": file_info.path, "target_path": None, "start_time": start_time}

    parts = parse_file_name(file_info.name)
    if parts is None:
        record.update(status="SKIPPED", message="File name does not match <table>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.zip",
                      end_time=datetime.now())
        return record

    target_folder = f"{target_base_path}{parts['mid_date']}/"
    record["target_path"] = f"{target_folder}{file_info.name}"
    try:
        dbutils.fs.mkdirs(target_folder)
        dbutils.fs.cp(file_info.path, record["target_path"])
        record.update(status="SUCCESS", message="Copied successfully")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record

## Stage 2 - Unzip

Unzip one copied file into `<unzip_tgt_path>/<mid_date>/<zip_name_without_.zip>/`.

Each zip gets its own sub-folder, so JSON files from different zips can never overwrite each other.

In [0]:
def unzip_zip_file(copy_record, unzip_base_path):
    start_time = datetime.now()
    file_name = copy_record["file_name"]
    zip_path  = copy_record["target_path"]
    unzip_base_path = unzip_base_path.rstrip("/") + "/"
    record = {"s_no": copy_record["s_no"], "stage": "UNZIP", "source": copy_record["source"],
              "file_name": file_name, "source_path": zip_path, "target_path": None, "start_time": start_time}

    mid_date = parse_file_name(file_name)["mid_date"]
    target_folder = f"{unzip_base_path}{mid_date}/{file_name.rsplit('.', 1)[0]}/"
    record["target_path"] = target_folder
    try:
        dbutils.fs.mkdirs(target_folder)
        with zipfile.ZipFile(zip_path.replace("dbfs:", ""), "r") as z:
            z.extractall(target_folder)
        record.update(status="SUCCESS", message="Extracted successfully")
    except zipfile.BadZipFile:
        record.update(status="FAILED", message="Invalid ZIP file")
    except Exception as e:
        record.update(status="FAILED", message=str(e))
    record["end_time"] = datetime.now()
    return record

## Stage 3 - Ingest JSON
Same logic as `json_large_files_ingestion_v1`: one metadata row per JSON file (schema captured as `VARIANT`), plus the new **`zip_file`** column.

In [0]:
def create_target_table(target):
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target} (
        src_system      STRING,
        file_id         BIGINT,      -- serial no.
        zip_file        STRING,      -- zip the JSON file
        file_name       STRING,
        file_date       TIMESTAMP,   -- modified time of the extracted file
        file_path       STRING,
        dt_datafile     STRING,        -- middle date from file name
        file_size_mb    DOUBLE,
        file_structure  VARIANT,
        table_name      STRING,      -- from file name
        created_by      STRING,
        created_on      TIMESTAMP
    ) USING DELTA
    """)
    # table created by an earlier version of the pipeline? add the new column
    if "zip_file" not in spark.table(target).columns:
        spark.sql(f"ALTER TABLE {target} ADD COLUMNS (zip_file STRING AFTER file_id)")


def list_json_files(path):
    """All .json files under path, including sub-folders."""
    out = []
    for item in dbutils.fs.ls(path):
        if item.isDir():
            out.extend(list_json_files(item.path))
        elif item.name.lower().endswith(".json"):
            out.append(item)
    return sorted(out, key=lambda x: x.path)


def log_load_event(run_id, ctl, zip_file, f, target, status, started, error=None):
    """Append one row to the load log. Logging must never break the load."""
    try:
        row = (
            run_id, ctl["zip_source"], ctl["unzip_tgt_path"], zip_file,
            f.name if f else None, f.path if f else None,
            f.size / (1024.0 * 1024.0) if f else None,
            int(f.modificationTime) if f else None,
            target, status,
            type(error).__name__ if error else None,
            (str(error)[:4000] + "\n" + traceback.format_exc()[-2000:]) if error else None,
            started, datetime.now(), EXECUTED_BY,
        )
        spark.createDataFrame([row], LOAD_LOG_SCHEMA).write.mode("append").saveAsTable(load_log_table)
    except Exception as log_err:
        print(f"WARNING: could not write load log row: {log_err}")


def load_file(f, target, zip_file):
    """Read one JSON file, capture its structure, write one metadata row to target."""
    parts = parse_file_name(f.name)
    if parts is None:
        raise ValueError(f"Bad file name '{f.name}'. Expected <table>_<yyyymmdd>_<yyyymmdd>_<yyyymmdd>_<hhmmss>.json")
    dt_datafile = datetime.strptime(parts["mid_date"], "%Y%m%d").strftime("%Y%m%d")

    file_structure = spark.read.option("multiLine", "true").json(f.path).schema.json()

    # replace any row from an earlier run of the same file, so re-runs never duplicate
    spark.sql(f"DELETE FROM {target} WHERE file_path = :p", args={"p": f.path})
    file_id = spark.sql(f"SELECT COALESCE(MAX(file_id), 0) + 1 FROM {target}").first()[0]

    (spark.createDataFrame([Row(
            src_system     = SRC_SYSTEM,
            file_id        = file_id,
            zip_file       = zip_file,
            file_name      = f.name,
            file_date      = datetime.fromtimestamp(f.modificationTime / 1000),
            file_path      = f.path,
            dt_datafile    = dt_datafile,
            file_size_mb   = f.size / (1024.0 * 1024.0),
            file_structure = file_structure,
            table_name     = parts["table_name"],
            created_by     = EXECUTED_BY,
        )])
        .withColumn("file_structure", F.parse_json("file_structure"))
        .withColumn("created_on", F.current_timestamp())
        .select("src_system", "file_id", "zip_file", "file_name", "file_date", "file_path", "dt_datafile",
                "file_size_mb", "file_structure", "table_name", "created_by", "created_on")
        .write.mode("append").saveAsTable(target))


def ingest_unzipped_zip(unzip_record, ctl, run_id):
    """Ingest every JSON file extracted from one zip. Returns {'SUCCESS': n, 'FAILED': n}."""
    target   = f"{ctl['tgt_catalog']}.{ctl['tgt_schema']}.{ctl['tgt_table']}"
    zip_file = unzip_record["file_name"]
    counts   = {"SUCCESS": 0, "FAILED": 0}

    started = datetime.now()
    try:
        create_target_table(target)
        files = list_json_files(unzip_record["target_path"])
        if not files:
            raise ValueError(f"No .json files found in {unzip_record['target_path']}")
    except Exception as e:
        log_load_event(run_id, ctl, zip_file, None, target, "FAILED", started, e)
        counts["FAILED"] += 1
        print(f"  [INGEST:FAILED] {zip_file}: {str(e)[:300]}")
        return counts

    for f in files:
        started = datetime.now()
        try:
            load_file(f, target, zip_file)
            log_load_event(run_id, ctl, zip_file, f, target, "SUCCESS", started)
            counts["SUCCESS"] += 1
            print(f"  [INGEST:SUCCESS] {f.name}")
        except Exception as e:
            log_load_event(run_id, ctl, zip_file, f, target, "FAILED", started, e)
            counts["FAILED"] += 1
            print(f"  [INGEST:FAILED] {f.name}: {str(e)[:300]}")
    return counts